# Piloto de fine-tuning — español sintético con Chatterbox V2

Chatterbox V2 es el profesor sin alucinaciones: en la escucha a ciegas sacó 4,25 de acento nativo y 5 de «sin alucinaciones». Parámetros por defecto (`exaggeration=0.5`, `cfg_weight=0.5`, `temperature=0.8`), los mismos de la criba.

Genera las **400 frases** de `piloto/frases_sinteticas.json` (unos 45 min de audio) con la voz de Naval,
usando `speaker_refs/naval/naval_ref02.wav` como referencia. Salida: `fine_tunned_naval/piloto/sintetico/`
en Drive, un `.wav` por frase (`s001` … `s400`).

**GPU:** T4 (o L4 si no hay T4). **Tiempo estimado:** unos 60 min.

Uso: celda 1, *Entorno de ejecución → Reiniciar sesión*, y el resto en orden. Si se corta, vuelve a empezar:
lo que ya esté en Drive no se genera otra vez. La última celda sube todo a Drive y desconecta la GPU.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
# Desde GitHub: la 0.1.7 de PyPI no tiene el parámetro t3_model. torchvision a juego con torch 2.6.0.
# El aviso rojo "pip's dependency resolver..." es normal.
!pip install -q "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@master" torchvision==0.21.0

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencia de Naval
import os, sys, subprocess
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun

comun.montar_drive()
comun.FRASES = Path("/content/repo/piloto/frases_sinteticas.json")
comun.SALIDA = comun.DRIVE / "piloto" / "sintetico"
ITEMS, SILENCIO_MS = comun.cargar_items()
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Generar
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
VARIANTE = "chatterbox_v2"
PARAMS = dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8)
modelo = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v2")
modelo.prepare_conditionals(REFS["naval"]["wav"], exaggeration=PARAMS["exaggeration"])

def sintetizar(texto, ref):
    return modelo.generate(texto, language_id="es", **PARAMS), modelo.sr

comun.guardar_entorno(VARIANTE, ["chatterbox-tts", "torch", "transformers", "numpy"])
comun.generar(VARIANTE, sintetizar, ITEMS, REFS, SILENCIO_MS, params={**PARAMS, "language_id": "es"})

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
filas = [json.loads(l) for l in (comun.SALIDA / "chatterbox_v2" / "registro.jsonl").read_text(encoding="utf-8").splitlines()]
ok = sum(f["ok"] for f in filas)
minutos = sum(f.get("segundos_audio", 0) for f in filas if f["ok"]) / 60
print(f"chatterbox_v2: {ok} clips correctos ({minutos:.0f} min), {len(filas) - ok} con error")
comun.desconectar()